In [1]:
import json
import os
import threading
import pandas as pd
import sys
sys.path.append('./')
from utils.MultiLabelPredictor import MultilabelPredictor

/home/massimo/Documents/stage/signature_inference/autogluon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bin_ground_truth_path = './simulations/ground_truth/bin_exposures.csv'
runs_path = './simulations/data/run_'
data_path = '/trinucleotides_counts_sampling_'
save_evaluation_path = './results/with_tissues_evaluations.json'
tissues_path = './simulations/ground_truth/tumor_site.csv'

In [3]:
bin_gt_df = pd.read_csv(bin_ground_truth_path)
bin_gt_df.head()

,Unnamed: 0,S1 (SBS1 - 0.99),S2 (SBS2 - 0.99),S3 (SBS3 - 0.97),S4 (SBS4 - 0.98),S5 (SBS5 - 0.98),S6 (SBS7a - 1.00),S7 (SBS7b - 0.96),S8 (SBS8 - 0.92),S9 (SBS9 - 0.94),...,S20 (SBS22 - 0.99),S21 (SBS23 - 0.94),S22 (SBS26 - 0.94),S23 (SBS28 - 0.96),S24 (SBS31 - 0.98),S25 (SBS32 - 0.94),S26 (SBS44 - 0.97),S27 (SBS88 - 0.92),S28 (SBS92 - 0.95),S29 (SBS97 - 0.95)
0,0009b464-b376-4fbc-8a56-da538269a02f,True,True,True,True,True,True,False,True,False,...,True,False,False,False,True,True,False,False,False,False
1,00493087-9d9d-40ca-86d5-936f1b951c93,True,False,False,False,True,False,False,False,False,...,False,True,True,False,False,False,False,True,False,False
2,00508f2b-36bf-44fc-b66b-97e1f3e40bfa,True,True,False,True,True,True,True,False,False,...,False,False,False,False,False,False,False,True,False,False
3,005794f1-5a87-45b5-9811-83ddf6924568,True,True,True,True,True,True,True,True,False,...,True,True,False,True,False,True,False,False,True,False
4,005e85a3-3571-462d-8dc9-2babfc7ace21,True,False,False,False,True,True,False,False,False,...,False,False,False,False,False,True,True,False,False,False


In [4]:
tissues_df = pd.read_csv(tissues_path)
tissues_df.head()

,Unnamed: 0,Organ,Cohort
0,0009b464-b376-4fbc-8a56-da538269a02f,Ovary,ICGC
1,00493087-9d9d-40ca-86d5-936f1b951c93,Central Nervous System,ICGC
2,00508f2b-36bf-44fc-b66b-97e1f3e40bfa,Neuroendocrine Tumors,ICGC
3,005794f1-5a87-45b5-9811-83ddf6924568,Kidney,ICGC
4,005e85a3-3571-462d-8dc9-2babfc7ace21,Prostate,ICGC


In [5]:
data_df = pd.read_csv('./simulations/data/run_1/trinucleotides_counts_sampling_0.1.csv')
data_df.head()

,Unnamed: 0,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,...,T[T>A]G,T[T>A]T,T[T>C]A,T[T>C]C,T[T>C]G,T[T>C]T,T[T>G]A,T[T>G]C,T[T>G]G,T[T>G]T
0,0009b464-b376-4fbc-8a56-da538269a02f,38,36,4,28,17,7,4,16,25,...,4,14,13,12,2,14,7,8,3,12
1,00493087-9d9d-40ca-86d5-936f1b951c93,3,5,0,2,3,0,0,1,5,...,1,3,2,2,4,3,0,0,0,2
2,00508f2b-36bf-44fc-b66b-97e1f3e40bfa,5,4,1,0,0,0,0,0,6,...,1,1,3,0,3,5,1,0,0,0
3,005794f1-5a87-45b5-9811-83ddf6924568,22,5,0,11,8,1,1,5,9,...,4,10,2,1,4,3,3,3,6,8
4,005e85a3-3571-462d-8dc9-2babfc7ace21,0,0,0,5,0,0,0,1,7,...,0,2,0,1,0,4,2,1,1,1


In [6]:
df = pd.merge(data_df, bin_gt_df, on='Unnamed: 0')
df.head()

,Unnamed: 0,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,...,S20 (SBS22 - 0.99),S21 (SBS23 - 0.94),S22 (SBS26 - 0.94),S23 (SBS28 - 0.96),S24 (SBS31 - 0.98),S25 (SBS32 - 0.94),S26 (SBS44 - 0.97),S27 (SBS88 - 0.92),S28 (SBS92 - 0.95),S29 (SBS97 - 0.95)
0,0009b464-b376-4fbc-8a56-da538269a02f,38,36,4,28,17,7,4,16,25,...,True,False,False,False,True,True,False,False,False,False
1,00493087-9d9d-40ca-86d5-936f1b951c93,3,5,0,2,3,0,0,1,5,...,False,True,True,False,False,False,False,True,False,False
2,00508f2b-36bf-44fc-b66b-97e1f3e40bfa,5,4,1,0,0,0,0,0,6,...,False,False,False,False,False,False,False,True,False,False
3,005794f1-5a87-45b5-9811-83ddf6924568,22,5,0,11,8,1,1,5,9,...,True,True,False,True,False,True,False,False,True,False
4,005e85a3-3571-462d-8dc9-2babfc7ace21,0,0,0,5,0,0,0,1,7,...,False,False,False,False,False,True,True,False,False,False


In [7]:
train_df = df.sample(frac=0.8, random_state=0)
test_df = df.drop(train_df.index)
print(train_df.shape)
train_df.head()

(14495, 126)


,Unnamed: 0,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,...,S20 (SBS22 - 0.99),S21 (SBS23 - 0.94),S22 (SBS26 - 0.94),S23 (SBS28 - 0.96),S24 (SBS31 - 0.98),S25 (SBS32 - 0.94),S26 (SBS44 - 0.97),S27 (SBS88 - 0.92),S28 (SBS92 - 0.95),S29 (SBS97 - 0.95)
16885,HMF003210A,3,1,0,1,0,1,1,3,3,...,False,False,False,False,False,True,False,True,False,False
6591,GEL-2361061-11,2,1,0,2,0,1,0,1,9,...,False,False,False,False,False,False,True,False,False,False
3639,GEL-2111089-11,7,5,1,6,3,1,0,4,5,...,True,False,False,False,False,True,False,True,True,False
11456,GEL-2764583-11,5,3,2,6,8,0,4,3,10,...,True,False,False,False,False,True,False,True,True,True
13818,GEL-2964830-11,14,13,0,7,15,2,0,4,15,...,False,False,False,False,False,False,False,True,False,False


In [8]:
print(test_df.shape)
test_df.head()

(3624, 126)


,Unnamed: 0,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,...,S20 (SBS22 - 0.99),S21 (SBS23 - 0.94),S22 (SBS26 - 0.94),S23 (SBS28 - 0.96),S24 (SBS31 - 0.98),S25 (SBS32 - 0.94),S26 (SBS44 - 0.97),S27 (SBS88 - 0.92),S28 (SBS92 - 0.95),S29 (SBS97 - 0.95)
2,00508f2b-36bf-44fc-b66b-97e1f3e40bfa,5,4,1,0,0,0,0,0,6,...,False,False,False,False,False,False,False,True,False,False
3,005794f1-5a87-45b5-9811-83ddf6924568,22,5,0,11,8,1,1,5,9,...,True,True,False,True,False,True,False,False,True,False
10,00c27940-c623-11e3-bf01-24c6515278c0,56,20,10,18,9,3,0,6,27,...,False,False,True,False,False,False,True,True,True,True
19,01dc6872-c623-11e3-bf01-24c6515278c0,19,6,1,11,4,2,1,6,23,...,False,False,True,False,True,False,False,True,True,True
27,030695f6-c623-11e3-bf01-24c6515278c0,22,16,5,12,9,7,1,9,20,...,False,False,True,False,True,False,False,True,True,True


In [9]:
labels = bin_gt_df.columns[1:]
problem_type = ['binary'] * len(labels)
time_limit = 5
# Create the model
predictor = MultilabelPredictor(labels=labels, problem_types=problem_type)
predictor.fit(train_df, time_limit=time_limit)

No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets.
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='best_quality'   : Maximize accuracy. Default time_limit=3600.
	presets='high_quality'   : Strong accuracy with fast inference speed. Default time_limit=3600.
	presets='good_quality'   : Good accuracy with very fast inference speed. Default time_limit=3600.
	presets='medium_quality' : Fast training time, ideal for initial prototyping.
Beginning AutoGluon training ... Time limit = 5s
AutoGluon will save models to "AutogluonModels/ag-20240910_151915/Predictor_S1 (SBS1 - 0.99)"
=================== System Info ===================
AutoGluon Version:  1.1.0
Python Version:     3.10.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #202405300957~1718348209~22.04~7817b67 SMP PREEMPT_DYNAMIC Mon J
CPU Count:          8
Memo

Fitting TabularPredictor for label: S1 (SBS1 - 0.99) ...


	Available Memory:                    10173.86 MB
	Train Data (Original)  Memory Usage: 11.62 MB (0.1% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (Count: 1): ['Unnamed: 0']
		These features were not used to generate any of the output features. Add a feature generator compatible with these features to utilize them.
		Features can also be unused if they carry very little information, such as being categorical but having almost entirely unique values or being duplicat

Fitting TabularPredictor for label: S2 (SBS2 - 0.99) ...


Automatically generating train/validation split with holdout_frac=0.1, Train Rows: 13045, Val Rows: 1450
User-specified model hyperparameters to be fit:
{
	'NN_TORCH': {},
	'GBM': [{'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}}, {}, 'GBMLarge'],
	'CAT': {},
	'XGB': {},
	'FASTAI': {},
	'RF': [{'criterion': 'gini', 'ag_args': {'name_suffix': 'Gini', 'problem_types': ['binary', 'multiclass']}}, {'criterion': 'entropy', 'ag_args': {'name_suffix': 'Entr', 'problem_types': ['binary', 'multiclass']}}, {'criterion': 'squared_error', 'ag_args': {'name_suffix': 'MSE', 'problem_types': ['regression', 'quantile']}}],
	'XT': [{'criterion': 'gini', 'ag_args': {'name_suffix': 'Gini', 'problem_types': ['binary', 'multiclass']}}, {'criterion': 'entropy', 'ag_args': {'name_suffix': 'Entr', 'problem_types': ['binary', 'multiclass']}}, {'criterion': 'squared_error', 'ag_args': {'name_suffix': 'MSE', 'problem_types': ['regression', 'quantile']}}],
	'KNN': [{'weights': 'uniform', 'ag_args': {'name_

...

Fitting TabularPredictor for label: S28 (SBS92 - 0.95) ...


	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (Count: 1): ['Unnamed: 0']
		These features were not used to generate any of the output features. Add a feature generator compatible with these features to utilize them.
		Features can also be unused if they carry very little information, such as being categorical but having almost entirely unique values or being duplicates of other features.
		These features do not need to be present at inference time.
		('object', []) : 1 | ['Unnamed: 0']
	Types of features in original data (raw dtype, special dtypes):
		('bool', []) : 27 | ['S1 (SBS1 - 0.99)', 'S2 (SBS2 - 0.99)', 'S3 (SBS3 - 0.97)', 'S4 (SBS4 - 0.98)', 'S5 (SBS5 - 0.98)', ...]
		('int', [])  : 96 | ['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T', 'A[C>G]A', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('int', [])       : 96 | ['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T', 'A[C>G]A', ...]
		('int', ['bool']) : 27 | ['S1 (S

Fitting TabularPredictor for label: S29 (SBS97 - 0.95) ...


	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Unused Original Features (Count: 1): ['Unnamed: 0']
		These features were not used to generate any of the output features. Add a feature generator compatible with these features to utilize them.
		Features can also be unused if they carry very little information, such as being categorical but having almost entirely unique values or being duplicates of other features.
		These features do not need to be present at inference time.
		('object', []) : 1 | ['Unnamed: 0']
	Types of features in original data (raw dtype, special dtypes):
		('bool', []) : 28 | ['S1 (SBS1 - 0.99)', 'S2 (SBS2 - 0.99)', 'S3 (SBS3 - 0.97)', 'S4 (SBS4 - 0.98)', 'S5 (SBS5 - 0.98)', ...]
		('int', [])  : 96 | ['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T', 'A[C>G]A', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('int', [])       : 96 | ['A[C>A]A', 'A[C>A]C', 'A[C>A]G', 'A[C>A]T', 'A[C>G]A', ...]
		('int', ['bool']) : 28 | ['S1 (S

MultilabelPredictor saved to disk. Load with: MultilabelPredictor.load('AutogluonModels/ag-20240910_151915')


In [10]:
evaluation = predictor.evaluate(test_df)

Evaluating TabularPredictor for label: S1 (SBS1 - 0.99) ...
Evaluating TabularPredictor for label: S2 (SBS2 - 0.99) ...
Evaluating TabularPredictor for label: S3 (SBS3 - 0.97) ...
Evaluating TabularPredictor for label: S4 (SBS4 - 0.98) ...
Evaluating TabularPredictor for label: S5 (SBS5 - 0.98) ...
Evaluating TabularPredictor for label: S6 (SBS7a - 1.00) ...
Evaluating TabularPredictor for label: S7 (SBS7b - 0.96) ...
Evaluating TabularPredictor for label: S8 (SBS8 - 0.92) ...
Evaluating TabularPredictor for label: S9 (SBS9 - 0.94) ...
Evaluating TabularPredictor for label: S10 (SBS10a - 1.00) ...
Evaluating TabularPredictor for label: S11 (SBS10d - 0.98) ...
Evaluating TabularPredictor for label: S12 (SBS11 - 0.99) ...
Evaluating TabularPredictor for label: S13 (SBS13 - 0.99) ...
Evaluating TabularPredictor for label: S14 (SBS14 - 0.98) ...
Evaluating TabularPredictor for label: S15 (SBS15 - 0.97) ...
Evaluating TabularPredictor for label: S16 (SBS17 - 0.99) ...
Evaluating TabularPred

In [11]:
x = test_df[0:1]
y = predictor.predict(x)

Predicting with TabularPredictor for label: S1 (SBS1 - 0.99) ...
Predicting with TabularPredictor for label: S2 (SBS2 - 0.99) ...
Predicting with TabularPredictor for label: S3 (SBS3 - 0.97) ...
Predicting with TabularPredictor for label: S4 (SBS4 - 0.98) ...
Predicting with TabularPredictor for label: S5 (SBS5 - 0.98) ...
Predicting with TabularPredictor for label: S6 (SBS7a - 1.00) ...
Predicting with TabularPredictor for label: S7 (SBS7b - 0.96) ...
Predicting with TabularPredictor for label: S8 (SBS8 - 0.92) ...
Predicting with TabularPredictor for label: S9 (SBS9 - 0.94) ...
Predicting with TabularPredictor for label: S10 (SBS10a - 1.00) ...
Predicting with TabularPredictor for label: S11 (SBS10d - 0.98) ...
Predicting with TabularPredictor for label: S12 (SBS11 - 0.99) ...
Predicting with TabularPredictor for label: S13 (SBS13 - 0.99) ...
Predicting with TabularPredictor for label: S14 (SBS14 - 0.98) ...
Predicting with TabularPredictor for label: S15 (SBS15 - 0.97) ...
Predictin

In [27]:
evaluations = predictor.evaluate(test_df)

Evaluating TabularPredictor for label: S1 (SBS1 - 0.99) ...
Evaluating TabularPredictor for label: S2 (SBS2 - 0.99) ...
Evaluating TabularPredictor for label: S3 (SBS3 - 0.97) ...
Evaluating TabularPredictor for label: S4 (SBS4 - 0.98) ...
Evaluating TabularPredictor for label: S5 (SBS5 - 0.98) ...
Evaluating TabularPredictor for label: S6 (SBS7a - 1.00) ...
Evaluating TabularPredictor for label: S7 (SBS7b - 0.96) ...
Evaluating TabularPredictor for label: S8 (SBS8 - 0.92) ...
Evaluating TabularPredictor for label: S9 (SBS9 - 0.94) ...
Evaluating TabularPredictor for label: S10 (SBS10a - 1.00) ...
Evaluating TabularPredictor for label: S11 (SBS10d - 0.98) ...
Evaluating TabularPredictor for label: S12 (SBS11 - 0.99) ...
Evaluating TabularPredictor for label: S13 (SBS13 - 0.99) ...
Evaluating TabularPredictor for label: S14 (SBS14 - 0.98) ...
Evaluating TabularPredictor for label: S15 (SBS15 - 0.97) ...
Evaluating TabularPredictor for label: S16 (SBS17 - 0.99) ...
Evaluating TabularPred

In [28]:
for e in evaluations:
    print(evaluations[e])
    break

{'accuracy': 0.9776490066225165, 'balanced_accuracy': 0.7953065134099617, 'mcc': 0.675457418853395, 'roc_auc': 0.9852290868454661, 'f1': 0.9884203002144388, 'precision': 0.9834992887624466, 'recall': 0.9933908045977011}


In [24]:
for evaluation in evaluations:
    target_class = predictor.get_predictor(evaluation)
    best_model = target_class.leaderboard(silent=True).iloc[0]['model']
    evaluations[evaluation]['best_model'] = best_model


In [25]:
for e in evaluations:
    print(evaluations[e])
    break

{'accuracy': 0.9776490066225165, 'balanced_accuracy': 0.7953065134099617, 'mcc': 0.675457418853395, 'roc_auc': 0.9852290868454661, 'f1': 0.9884203002144388, 'precision': 0.9834992887624466, 'recall': 0.9933908045977011, 'best_model': 'LightGBMXT'}
